# Student Wellbeing Classification

This notebook shows the full workflow cell by cell:

- binary classification: normal vs abnormal
- subtype classification: sleepy vs stress
- hierarchical classification: normal vs sleepy vs stress
- direct_3class_classification: normal vs sleepy vs stress without anomalous.csv

All final evaluations use Leave-One-Subject-Out cross-validation using the participant/source column. Random row splitting is not used.


In [1]:
from pathlib import Path
import os
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

from wellbeing_hierarchical_classification import (
    append_dataset_summary,
    append_interpretation,
    append_model_summary,
    best_predictions,
    build_binary_dataset,
    build_direct_3class_dataset,
    build_hierarchical_test_dataset,
    build_hierarchical_vs_direct_comparison,
    build_subtype_dataset,
    coerce_feature_columns,
    direct_3class_classification,
    evaluate_flat_task,
    evaluate_hierarchical_task,
    format_table,
    infer_group_column,
    inspect_group_candidates,
    load_datasets,
    model_factories,
    prepare_results_dirs,
    pivot_counts,
    resolve_csv_dir,
    save_confusion_matrix_png,
    save_feature_importance_png,
    save_random_forest_feature_importance,
    validate_anomalous_combination,
    validate_features,
)

DATA_DIR = Path('.')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIRS = prepare_results_dirs(RESULTS_DIR)

MPL_CONFIG_DIR = RESULTS_DIR / '.matplotlib'
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CONFIG_DIR.resolve()))

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 100)

csv_dir = resolve_csv_dir(DATA_DIR)
print(f'CSV directory: {csv_dir}')
print(f'Results directory: {RESULTS_DIR.resolve()}')
print('Organized result folders:')
for key, path in RESULT_DIRS.items():
    if key != 'root':
        print(f'- {key}: {path.resolve()}')


FileNotFoundError: Could not find normal.csv, anomalous.csv, sleepy.csv, stress.csv in /Users/mosi_dev/Developer/university/nearable-project/Scripts/ML or /Users/mosi_dev/Developer/university/nearable-project/Scripts/ML/data/supervised.

## 1. Load CSV Files

The direct 3-class experiment will not use `anomalous.csv`, but the hierarchical binary model needs it and we still validate that it duplicates sleepy + stress.


In [ ]:
raw_datasets = load_datasets(csv_dir)

for name, df in raw_datasets.items():
    print()
    print(f'{name}.csv: {df.shape[0]} rows x {df.shape[1]} columns')
    print(list(df.columns))



normal.csv: 218 rows x 21 columns
['window_start', 'video_source', 'p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD', 'label']

anomalous.csv: 431 rows x 21 columns
['window_start', 'video_source', 'p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD', 'label']

sleepy.csv: 213 rows x 21 columns
['window_start', 'video_source', 'p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_c

## 2. Inspect Participant / Source Columns

This checks possible grouping columns and infers the column used for Leave-One-Subject-Out CV.


In [ ]:
group_inspection = inspect_group_candidates(raw_datasets)
group_col = infer_group_column(raw_datasets)

print(f'Inferred group column: {group_col}')
for file_name, columns in group_inspection.items():
    print()
    print(f'{file_name}.csv candidate group columns:')
    if not columns:
        print('  none detected')
    for column, values in columns.items():
        print(f'  {column}: {values}')


Inferred group column: video_source

normal.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']

anomalous.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']

sleepy.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']

stress.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']


## 3. Validate Sensor Features and Missing Values

Only numerical sensor features are used. Identifiers such as `video_source`, names, filenames, `window_start`, and labels are excluded from the model features. Median imputation happens inside each scikit-learn pipeline.


In [ ]:
features = validate_features(raw_datasets)
datasets = coerce_feature_columns(raw_datasets, features)

print('Selected numerical feature columns:')
print(features)
print()
print('Note: the file uses RMMSD; scientifically this refers to RMSSD.')
print()
print('Missing values in selected features:')
for name, df in datasets.items():
    missing = df.loc[:, features].isna().sum()
    print()
    print(f'{name}.csv')
    print(missing.to_string())


Selected numerical feature columns:
['p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD']

Note: the file uses RMMSD; scientifically this refers to RMSSD.

Missing values in selected features:

normal.csv
p_shoulder_mean        0
p_shoulder_std         0
p_shoulder_bad_pct     0
p_head_tilt_mean       0
p_head_tilt_std        0
p_head_tilt_bad_pct    0
p_fhp_mean             0
p_fhp_std              0
p_fhp_bad_pct          0
b_ear_mean             0
b_ear_std              0
b_perclos              0
b_blink_count          0
b_blink_freq           0
b_ibi_mean             0
b_ibi_std              0
HR                     0
RMMSD                  0

anomalous.csv
p_shoulder_mean        0
p_shoulder_std         0
p_shoulder_bad_pct     0
p_head_tilt_mean       0
p_h

## 4. Validate anomalous.csv

This checks whether `anomalous.csv` is exactly `sleepy.csv + stress.csv` after relabeling to anomalous.


In [ ]:
validation = validate_anomalous_combination(datasets, group_col)
for key, value in validation.items():
    print(f'{key}: {value}')


anomalous_rows: 431
sleepy_rows: 213
stress_rows: 218
sleepy_plus_stress_rows: 431
same_columns: True
exact_ordered_match_after_relabel: True
exact_unordered_match_after_relabel: True
anomalous_by_group: {'Alessia': 110, 'Alvaro': 111, 'Beatrice': 108, 'Mo': 102}
sleepy_plus_stress_by_group: {'Alessia': 110, 'Alvaro': 111, 'Beatrice': 108, 'Mo': 102}


## 5. Build Modeling Datasets

- Binary: normal + anomalous
- Subtype: sleepy + stress
- Hierarchical final labels: normal + sleepy + stress
- Direct 3-class: normal + sleepy + stress only, excluding anomalous.csv to avoid duplication


In [ ]:
binary_df = build_binary_dataset(datasets, group_col)
subtype_df = build_subtype_dataset(datasets, group_col)
final_df = build_hierarchical_test_dataset(datasets, group_col)
direct_df = build_direct_3class_dataset(datasets, group_col)

print('Binary rows per participant/group per class:')
print(format_table(pivot_counts(binary_df, group_col)))
print()
print('Subtype rows per participant/group per class:')
print(format_table(pivot_counts(subtype_df, group_col)))
print()
print('Hierarchical final-label rows per participant/group per class:')
print(format_table(pivot_counts(final_df, group_col)))
print()
print('Direct 3-class rows per participant/group per class:')
print(format_table(pivot_counts(direct_df, group_col)))


Binary rows per participant/group per class:
video_source  abnormal  normal  All
     Alessia       110      54  164
      Alvaro       111      54  165
    Beatrice       108      55  163
          Mo       102      55  157
         All       431     218  649

Subtype rows per participant/group per class:
video_source  sleepy  stress  All
     Alessia      55      55  110
      Alvaro      56      55  111
    Beatrice      54      54  108
          Mo      48      54  102
         All     213     218  431

Hierarchical final-label rows per participant/group per class:
video_source  normal  sleepy  stress  All
     Alessia      54      55      55  164
      Alvaro      54      56      55  165
    Beatrice      55      54      54  163
          Mo      55      48      54  157
         All     218     213     218  649

Direct 3-class rows per participant/group per class:
video_source  normal  sleepy  stress  All
     Alessia      54      55      55  164
      Alvaro      54      56      

## 6. Define Models

Logistic Regression and SVM include `StandardScaler`. All models include median imputation inside the pipeline.


In [ ]:
models = model_factories()
binary_labels = ['normal', 'abnormal']
subtype_labels = ['sleepy', 'stress']
hierarchical_labels = ['normal', 'sleepy', 'stress']
direct_labels = ['normal', 'sleepy', 'stress']

for model_name in models:
    print(f'- {model_name}')


- Logistic Regression
- SVM
- Random Forest
- Gradient Boosting


## 7. Binary Classification: Normal vs Abnormal


In [ ]:
binary_results, binary_best, binary_predictions = evaluate_flat_task(
    'binary',
    binary_df,
    features,
    group_col,
    binary_labels,
    models,
)
binary_results.to_csv(RESULT_DIRS['binary'] / 'binary_results.csv', index=False)

binary_overall = (
    binary_results[binary_results['row_type'] == 'overall']
    .loc[:, ['model', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'precision_macro', 'recall_macro']]
    .sort_values(['macro_f1', 'balanced_accuracy'], ascending=False)
)
print('Overall binary model ranking:')
print(format_table(binary_overall))
print()
print(f"Selected binary model: {binary_best['model']}")
print('Per-fold results for selected binary model:')
print(format_table(
    binary_results[(binary_results['row_type'] == 'fold') & (binary_results['model'] == binary_best['model'])]
    .loc[:, ['fold', 'heldout_group', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']]
))

binary_selected = best_predictions(binary_best, binary_predictions)
print()
print('Classification report:')
print(classification_report(binary_selected['y_true'], binary_selected['y_pred'], labels=binary_labels, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(binary_selected['y_true'], binary_selected['y_pred'], labels=binary_labels))

save_confusion_matrix_png(
    binary_selected['y_true'],
    binary_selected['y_pred'],
    binary_labels,
    f"Binary LOSO Confusion Matrix: {binary_best['model']}",
    RESULT_DIRS['binary'] / 'binary_confusion_matrix.png',
)
save_random_forest_feature_importance(
    binary_df,
    features,
    RESULT_DIRS['binary'] / 'feature_importance_binary.csv',
    models['Random Forest'],
    str(binary_best['model']),
)


Overall binary model ranking:
              model  accuracy  balanced_accuracy  macro_f1  weighted_f1  precision_macro  recall_macro
      Random Forest  0.956857           0.958450  0.952279     0.957130         0.946975      0.958450
  Gradient Boosting  0.947612           0.948089  0.942054     0.947944         0.936874      0.948089
                SVM  0.926040           0.942049  0.920660     0.927441         0.909630      0.942049
Logistic Regression  0.887519           0.915313  0.882030     0.890382         0.874570      0.915313

Selected binary model: Random Forest
Per-fold results for selected binary model:
 fold heldout_group  accuracy  balanced_accuracy  macro_f1  weighted_f1
    1       Alessia  1.000000           1.000000  1.000000     1.000000
    2        Alvaro  0.951515           0.925926  0.942609     0.950419
    3      Beatrice  0.993865           0.995370  0.993170     0.993878
    4            Mo  0.878981           0.906863  0.875005     0.881679

Classificati

## 8. Subtype Classification: Sleepy vs Stress


In [ ]:
subtype_results, subtype_best, subtype_predictions = evaluate_flat_task(
    'subtype',
    subtype_df,
    features,
    group_col,
    subtype_labels,
    models,
)
subtype_results.to_csv(RESULT_DIRS['subtype'] / 'subtype_results.csv', index=False)

subtype_overall = (
    subtype_results[subtype_results['row_type'] == 'overall']
    .loc[:, ['model', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'precision_macro', 'recall_macro']]
    .sort_values(['macro_f1', 'balanced_accuracy'], ascending=False)
)
print('Overall subtype model ranking:')
print(format_table(subtype_overall))
print()
print(f"Selected subtype model: {subtype_best['model']}")
print('Per-fold results for selected subtype model:')
print(format_table(
    subtype_results[(subtype_results['row_type'] == 'fold') & (subtype_results['model'] == subtype_best['model'])]
    .loc[:, ['fold', 'heldout_group', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']]
))

subtype_selected = best_predictions(subtype_best, subtype_predictions)
print()
print('Classification report:')
print(classification_report(subtype_selected['y_true'], subtype_selected['y_pred'], labels=subtype_labels, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(subtype_selected['y_true'], subtype_selected['y_pred'], labels=subtype_labels))

save_confusion_matrix_png(
    subtype_selected['y_true'],
    subtype_selected['y_pred'],
    subtype_labels,
    f"Subtype LOSO Confusion Matrix: {subtype_best['model']}",
    RESULT_DIRS['subtype'] / 'subtype_confusion_matrix.png',
)
save_random_forest_feature_importance(
    subtype_df,
    features,
    RESULT_DIRS['subtype'] / 'feature_importance_subtype.csv',
    models['Random Forest'],
    str(subtype_best['model']),
)


Overall subtype model ranking:
              model  accuracy  balanced_accuracy  macro_f1  weighted_f1  precision_macro  recall_macro
      Random Forest  1.000000           1.000000  1.000000     1.000000         1.000000      1.000000
  Gradient Boosting  0.986079           0.986185  0.986079     0.986079         0.986143      0.986185
Logistic Regression  0.972158           0.972154  0.972154     0.972158         0.972154      0.972154
                SVM  0.923434           0.923774  0.923407     0.923391         0.924725      0.923774

Selected subtype model: Random Forest
Per-fold results for selected subtype model:
 fold heldout_group  accuracy  balanced_accuracy  macro_f1  weighted_f1
    1       Alessia       1.0                1.0       1.0          1.0
    2        Alvaro       1.0                1.0       1.0          1.0
    3      Beatrice       1.0                1.0       1.0          1.0
    4            Mo       1.0                1.0       1.0          1.0

Classific

## 9. Hierarchical Classification

The binary model predicts normal vs abnormal first. If abnormal is predicted, the subtype model predicts sleepy vs stress.


In [ ]:
hierarchical_results, hierarchical_best, hierarchical_predictions = evaluate_hierarchical_task(
    binary_df,
    subtype_df,
    final_df,
    features,
    group_col,
    models,
    models,
    hierarchical_labels,
)
hierarchical_results.to_csv(RESULT_DIRS['hierarchical'] / 'hierarchical_results.csv', index=False)

hierarchical_overall = (
    hierarchical_results[hierarchical_results['row_type'] == 'overall']
    .loc[:, ['model', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'precision_macro', 'recall_macro']]
    .sort_values(['macro_f1', 'balanced_accuracy'], ascending=False)
)
print('Top hierarchical model combinations:')
print(format_table(hierarchical_overall.head(10)))
print()
print(f"Selected hierarchical combination: {hierarchical_best['model']}")
print('Per-fold results for selected hierarchical combination:')
print(format_table(
    hierarchical_results[(hierarchical_results['row_type'] == 'fold') & (hierarchical_results['model'] == hierarchical_best['model'])]
    .loc[:, ['fold', 'heldout_group', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']]
))

hierarchical_selected = best_predictions(hierarchical_best, hierarchical_predictions)
print()
print('Classification report:')
print(classification_report(hierarchical_selected['y_true'], hierarchical_selected['y_pred'], labels=hierarchical_labels, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(hierarchical_selected['y_true'], hierarchical_selected['y_pred'], labels=hierarchical_labels))

save_confusion_matrix_png(
    hierarchical_selected['y_true'],
    hierarchical_selected['y_pred'],
    hierarchical_labels,
    f"Hierarchical LOSO Confusion Matrix: {hierarchical_best['model']}",
    RESULT_DIRS['hierarchical'] / 'hierarchical_confusion_matrix.png',
)


Top hierarchical model combinations:
                                   model  accuracy  balanced_accuracy  macro_f1  weighted_f1  precision_macro  recall_macro
          Random Forest -> Random Forest  0.956857           0.957007  0.957293     0.957074         0.958308      0.957007
    Random Forest -> Logistic Regression  0.949153           0.949182  0.949648     0.949446         0.950953      0.949182
      Random Forest -> Gradient Boosting  0.949153           0.949326  0.949396     0.949188         0.950272      0.949326
      Gradient Boosting -> Random Forest  0.947612           0.947833  0.948156     0.947866         0.949199      0.947833
  Gradient Boosting -> Gradient Boosting  0.939908           0.940152  0.940237     0.939958         0.941117      0.940152
Gradient Boosting -> Logistic Regression  0.938367           0.938443  0.939047     0.938782         0.940553      0.938443
                    SVM -> Random Forest  0.926040           0.926318  0.926954     0.926538   

## 10. direct_3class_classification

This direct 3-class experiment predicts `normal`, `sleepy`, or `stress` in one model. It uses only `normal.csv`, `sleepy.csv`, and `stress.csv`. It does not use `anomalous.csv`.


In [ ]:
direct_results, direct_best, direct_predictions = direct_3class_classification(
    direct_df,
    features,
    group_col,
    models,
    direct_labels,
)
direct_results.to_csv(RESULT_DIRS['direct_3class'] / 'direct_3class_results.csv', index=False)
direct_results[direct_results['row_type'] == 'fold'].to_csv(
    RESULT_DIRS['direct_3class'] / 'direct_3class_per_fold_results.csv',
    index=False,
)

direct_overall = (
    direct_results[direct_results['row_type'] == 'overall']
    .loc[:, ['model', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'precision_macro', 'recall_macro']]
    .sort_values(['macro_f1', 'balanced_accuracy'], ascending=False)
)
print('Overall direct 3-class model ranking:')
print(format_table(direct_overall))
print()
print(f"Selected direct 3-class model: {direct_best['model']}")
print('Per-fold results for selected direct model:')
print(format_table(
    direct_results[(direct_results['row_type'] == 'fold') & (direct_results['model'] == direct_best['model'])]
    .loc[:, ['fold', 'heldout_group', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']]
))

direct_selected = best_predictions(direct_best, direct_predictions)
direct_report_text = classification_report(
    direct_selected['y_true'],
    direct_selected['y_pred'],
    labels=direct_labels,
    zero_division=0,
)
direct_cm = confusion_matrix(
    direct_selected['y_true'],
    direct_selected['y_pred'],
    labels=direct_labels,
)
print()
print('Classification report:')
print(direct_report_text)
print('Confusion matrix:')
print(direct_cm)

save_confusion_matrix_png(
    direct_selected['y_true'],
    direct_selected['y_pred'],
    direct_labels,
    f"Direct 3-Class LOSO Confusion Matrix: {direct_best['model']}",
    RESULT_DIRS['direct_3class'] / 'direct_3class_confusion_matrix.png',
)
report_lines = [
    f"Selected direct 3-class model: {direct_best['model']}",
    'Selection criterion: highest macro F1, then balanced accuracy, then weighted F1 and accuracy.',
    '',
    'Classification report:',
    direct_report_text,
    'Confusion matrix:',
    str(direct_cm),
]
(RESULT_DIRS['direct_3class'] / 'direct_3class_classification_report.txt').write_text(
    chr(10).join(report_lines) + chr(10),
    encoding='utf-8',
)

direct_importances = save_random_forest_feature_importance(
    direct_df,
    features,
    RESULT_DIRS['direct_3class'] / 'direct_3class_feature_importance.csv',
    models['Random Forest'],
    str(direct_best['model']),
)
save_feature_importance_png(
    direct_importances,
    RESULT_DIRS['direct_3class'] / 'direct_3class_feature_importance.png',
    'Direct 3-Class Random Forest Feature Importance',
)
print()
print('Saved direct 3-class CSV, report, confusion matrix, and feature importance files.')


Overall direct 3-class model ranking:
              model  accuracy  balanced_accuracy  macro_f1  weighted_f1  precision_macro  recall_macro
      Random Forest  0.950693           0.950855  0.951070     0.950857         0.952044      0.950855
Logistic Regression  0.929122           0.929053  0.930300     0.930102         0.940574      0.929053
  Gradient Boosting  0.921418           0.921982  0.921516     0.920966         0.927804      0.921982
                SVM  0.895223           0.895845  0.892392     0.891933         0.906169      0.895845

Selected direct 3-class model: Random Forest
Per-fold results for selected direct model:
 fold heldout_group  accuracy  balanced_accuracy  macro_f1  weighted_f1
    1       Alessia  1.000000           1.000000  1.000000     1.000000
    2        Alvaro  0.945455           0.944444  0.944148     0.944646
    3      Beatrice  0.993865           0.993827  0.993882     0.993864
    4            Mo  0.859873           0.860340  0.861357     0.8588

## 11. Hierarchical vs Direct 3-Class Comparison


In [ ]:
comparison = build_hierarchical_vs_direct_comparison(hierarchical_best, direct_best)
comparison.to_csv(RESULT_DIRS['comparison'] / 'hierarchical_vs_direct_3class_comparison.csv', index=False)
print(format_table(comparison))


     approach                          model  accuracy  balanced_accuracy  macro_f1  weighted_f1
 hierarchical Random Forest -> Random Forest  0.956857           0.957007  0.957293     0.957074
direct_3class                  Random Forest  0.950693           0.950855  0.951070     0.950857


## 12. Final Summary and Interpretation

This writes `results/summary/final_summary.txt` and prints the interpretation section.


In [ ]:
summary_lines = []
append_dataset_summary(
    summary_lines,
    datasets,
    group_col,
    group_inspection,
    validation,
    features,
    binary_df,
    subtype_df,
    final_df,
    direct_df,
)
append_model_summary(
    summary_lines,
    'BINARY: NORMAL VS ABNORMAL',
    binary_results,
    binary_best,
    binary_predictions,
    binary_labels,
)
append_model_summary(
    summary_lines,
    'SUBTYPE: SLEEPY VS STRESS',
    subtype_results,
    subtype_best,
    subtype_predictions,
    subtype_labels,
)
append_model_summary(
    summary_lines,
    'HIERARCHICAL: NORMAL, SLEEPY, STRESS',
    hierarchical_results,
    hierarchical_best,
    hierarchical_predictions,
    hierarchical_labels,
)
append_model_summary(
    summary_lines,
    'DIRECT 3-CLASS: NORMAL, SLEEPY, STRESS',
    direct_results,
    direct_best,
    direct_predictions,
    direct_labels,
)
append_interpretation(
    summary_lines,
    binary_best,
    subtype_best,
    hierarchical_best,
    direct_best,
    hierarchical_predictions,
    direct_predictions,
    hierarchical_labels,
    direct_labels,
    hierarchical_results,
    direct_results,
    comparison,
)

summary_text = chr(10).join(summary_lines)
summary_path = RESULT_DIRS['summary'] / 'final_summary.txt'
summary_path.write_text(summary_text + chr(10), encoding='utf-8')

interpretation_start = summary_text.index('SHORT INTERPRETATION')
print(summary_text[interpretation_start:])
print()
print(f'Saved final summary to: {summary_path.resolve()}')
print(f'Saved outputs in: {RESULTS_DIR.resolve()}')


SHORT INTERPRETATION
These results are proof-of-concept only. Leave-One-Subject-Out is the right evaluation choice here, but 4 participants is not enough for strong generalization claims.
Binary normal vs abnormal worked best with Random Forest (macro F1=0.952, balanced accuracy=0.958).
Sleepy vs stress worked best with Random Forest (macro F1=1.000, balanced accuracy=1.000).
The full hierarchical system worked best with Random Forest -> Random Forest (macro F1=0.957, balanced accuracy=0.957, weighted F1=0.957).
The direct 3-class system worked best with Random Forest (macro F1=0.951, balanced accuracy=0.951, weighted F1=0.951).

Hierarchical vs direct 3-class comparison:
     approach                          model  accuracy  balanced_accuracy  macro_f1  weighted_f1
 hierarchical Random Forest -> Random Forest  0.956857           0.957007  0.957293     0.957074
direct_3class                  Random Forest  0.950693           0.950855  0.951070     0.950857
Based mainly on macro F1, th